# 跨链协议（Cross-Chain Protocols）— 知识学习 Notebook

**分类**：密码协议设计专项  
**难度**：中等（建议 8 小时）  
**前置要求**：区块链基础、哈希函数、数字签名、Merkle 树、智能合约入门  

---

## 学习目标
- 理解跨链协议的背景与核心目标（互操作性、资产转移、消息传递、状态验证）
- 掌握三大跨链架构模型（HTLC 原子交换 / 中继链轻客户端 / 锚定与桥接）
- 能用 Python **模拟**跨链关键机制：HTLC 原子交换、Merkle 证明验证、桥接锁定-铸造、侧链 Peg-in/Peg-out
- 认识常见安全威胁与防御机制

> 本 Notebook 以“**去信任化（trustless）**”为核心诉求：尽量不依赖中心化中介，通过**可验证性**与**原子性**来保证跨链安全。


## 使用方式与学习节奏（建议）
你可以按“学习步骤 1 → 9”顺序完成，每一步包含：

- **核心概念**（Markdown）
- **关键公式/逻辑**（Markdown）
- **可运行代码**（Python）
- **小测与思考**（自检）

> 你可以边学边改代码：例如调整 `time_lock`、验证者阈值 `t`、Merkle 树叶子数量等，观察协议性质如何变化。


---
# Step 1：理解跨链协议背景与核心目标

### 1.1 区块链“孤岛问题”为什么出现？
不同链在以下方面存在差异，导致资产、状态无法直接互通：

- **共识机制**：PoW / PoS / BFT 等，最终性与重组（reorg）特性不同  
- **地址空间与账户模型**：UTXO vs Account  
- **加密算法与签名体系**：ECDSA/EdDSA 等  
- **虚拟机与智能合约执行环境**：EVM/WASM/自研 VM  
- **状态表示与数据结构**：不同的 state root、交易编码方式等  

### 1.2 跨链协议想实现什么？
跨链协议的核心目标（常见四件事）：

1. **互操作性（Interoperability）**：链与链之间能理解彼此的证明与消息  
2. **资产转移（Asset Transfer）**：A 链资产能在 B 链被“等价表示/可赎回”  
3. **状态证明与验证（State Proof & Verification）**：B 链能验证 A 链某状态/事件确实发生  
4. **消息传递（Message Passing）**：跨链调用、跨链治理、跨链合约协作  

### 1.3 数学基础与技术要求如何对应？
- **资产原子交换** → 典型依赖 **哈希时间锁 HTLC**（Hash + Time）  
- **状态验证** → 典型依赖 **Merkle 树证明** + **区块头可验证性**（签名/共识最终性）  
- **去信任化桥** → 轻客户端验证、门限签名、多方验证等  

> 重点：跨链安全往往归结为两点：  
> 1）**可验证性**：消息/状态能被接收链验证  
> 2）**原子性**：跨链动作要么全部发生，要么都不发生（或可安全回滚）


### 自检小测
1. 为什么不同链的共识差异会影响跨链？（提示：最终性、重组）  
2. “去信任化”跨链与“托管式桥”最大的区别是什么？  
3. 资产转移与消息传递在安全侧分别最依赖什么数学工具？


---
# Step 2：学习跨链三大架构模型

下面是跨链协议最常见的三大模型。你可以把它们当作三种“设计范式”。

## 2.1 HTLC 型（原子交换）
**核心机制**：哈希锁 + 时间锁 → 保证原子性  
**代表**：Atomic Swap、Lightning Network（链下也会用到类似思想）  
**适用**：点对点资产交换、小额快速交换、无需长期跨链状态跟踪  

## 2.2 中继链/轻客户端型（跨链消息与状态验证）
**核心机制**：接收链运行“发送链轻客户端”，验证对方区块头与 Merkle 证明  
**代表**：Cosmos IBC、Polkadot（中继链验证平行链块头）  
**适用**：规模化多链互通、跨链消息传递、跨链合约组合  

## 2.3 锚定与桥接型（Lock-Mint / Burn-Unlock）
**核心机制**：在链 A **锁定**资产 → 在链 B **铸造映射资产**；反向赎回则销毁映射资产  
**代表**：WBTC、Avalanche Bridge、LayerZero（不同实现差异很大）  
**适用**：资产“流动性迁移”、跨生态资产使用（DeFi 等）

---

### 对比记忆
- **HTLC**：靠哈希 + 时间锁保证“**原子交换**”  
- **轻客户端/中继链**：靠“**可验证证明**”保证消息与状态可信  
- **桥（Lock-Mint）**：靠验证者/门限签名/轻客户端证明，保证“锁定 ↔ 铸造”的一致性

> 经验口诀：  
> 小额快速交换 → HTLC  
> 多链规模互通 → 中继链/IBC  
> 资产跨生态流动 → 桥（但要特别关注安全）


---
# Step 3：深入 HTLC 原子交换机制

## 3.1 参与方
- Alice：持有链 A 资产  
- Bob：持有链 B 资产  

## 3.2 协议流程（高层）
1. Alice 生成随机秘密 `s`，计算哈希锁 `h = H(s)`  
2. Alice 在链 A 创建 HTLC：  
   - 若 Bob 提供 `s`（即 `H(s)=h`），则 Bob 可领取 Alice 的资产  
   - 否则超时 `T_A` 后 Alice 可退款  
3. Bob 在链 B 创建 HTLC：  
   - 同样的哈希锁 `h`  
   - 超时 `T_B`（**必须满足** `T_B < T_A`）以保证 Bob 来得及用 `s` 领取  
4. Alice 在链 B 用 `s` 领取 Bob 的资产（公开 `s`）  
5. Bob 观察到 `s` 后，在链 A 用 `s` 领取 Alice 的资产  

## 3.3 安全直觉
- **哈希不可逆**：在 Alice 未公开 `s` 前，Bob 无法推导 `s`  
- **时间锁**：若流程中断，双方都能在超时后退款  
- 原子性条件可写成：  
  $$
(A\_unlocks \Rightarrow B\_unlocks) \lor (timeout)
$$
- **关键设计**：`T_B < T_A`，否则 Alice 可能在链 B 解锁后，Bob 因超时来不及在链 A 解锁

下面我们会用 Python 模拟一个最小 HTLC 原子交换（不依赖真实链）。


## 3.4 Python：哈希锁与时间锁基础

In [ ]:
import hashlib, secrets, time
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional, Tuple

def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

# 生成秘密 s 和哈希锁 h
s = secrets.token_bytes(32)
h = sha256(s)
h, len(s)


你可以把 `s` 理解为“预映像 preimage”，`h` 是公开的哈希承诺。

---
# Step 4：掌握轻客户端验证与 Merkle 证明（中继链模型基础）

## 4.1 核心思想
接收链不需要同步发送链全部区块数据，只需：

- 保存对方链的**区块头**（或其可验证摘要）
- 对跨链消息 `m`，附带 Merkle 证明 `π`
- 验证：`VerifyProof(Root_A, m, π) = 1`

## 4.2 Merkle 证明验证逻辑（伪代码）
```js
function verifyMerkleProof(root, data, path) {
  let hash = sha256(data);
  for (const step of path) {
    if (step.left) hash = sha256(step.hash + hash);
    else          hash = sha256(hash + step.hash);
  }
  return hash === root;
}
```

> 轻客户端的优势：验证成本低（只验证区块头 + 路径），适合跨链消息与状态验证。


## 4.3 Python：构建 Merkle 树、生成证明、验证证明

In [ ]:
from typing import NamedTuple

class MerkleStep(NamedTuple):
    left: bool     # True 表示 sibling 在左边
    sibling: str   # sibling hash (hex)

def merkle_root(leaves: List[bytes]) -> str:
    # 叶子哈希
    layer = [sha256(x) for x in leaves]
    if not layer:
        return sha256(b"")
    # 逐层合并（若奇数个，复制最后一个）
    while len(layer) > 1:
        if len(layer) % 2 == 1:
            layer.append(layer[-1])
        nxt = []
        for i in range(0, len(layer), 2):
            nxt.append(sha256((layer[i] + layer[i+1]).encode()))
        layer = nxt
    return layer[0]

def merkle_proof(leaves: List[bytes], index: int) -> Tuple[str, List[MerkleStep], str]:
    # 返回 (root, path, leaf_hash)
    hashes = [sha256(x) for x in leaves]
    if index < 0 or index >= len(hashes):
        raise IndexError("index out of range")
    path: List[MerkleStep] = []
    layer = hashes[:]
    idx = index
    while len(layer) > 1:
        if len(layer) % 2 == 1:
            layer.append(layer[-1])
        sibling_idx = idx ^ 1
        left = (sibling_idx < idx)  # sibling 在左?
        path.append(MerkleStep(left=left, sibling=layer[sibling_idx]))
        # 合并到上一层
        nxt = []
        for i in range(0, len(layer), 2):
            nxt.append(sha256((layer[i] + layer[i+1]).encode()))
        layer = nxt
        idx //= 2
    root = layer[0]
    return root, path, hashes[index]

def verify_merkle_proof(root: str, leaf_hash: str, path: List[MerkleStep]) -> bool:
    h = leaf_hash
    for step in path:
        if step.left:
            h = sha256((step.sibling + h).encode())
        else:
            h = sha256((h + step.sibling).encode())
    return h == root

# Demo
leaves = [f"tx_{i}".encode() for i in range(7)]
root, path, leaf_hash = merkle_proof(leaves, index=3)
root2 = merkle_root(leaves)
print("Root一致:", root == root2)
print("验证结果:", verify_merkle_proof(root, leaf_hash, path))


### 4.4 思考题
- 如果发送链出现重组（reorg），区块头的 Root 变化会怎样影响跨链证明？
- 为什么轻客户端通常要跟踪“最终性（finality）”或设置确认数？

---
# Step 5：学习锚定与桥接模型（Lock-Mint / Burn-Unlock）

## 5.1 核心流程
以“链 A 锁定 1 BTC，链 B 铸造 1 WBTC”为例：

### 正向转移（A → B）
1. 用户在链 A 调用桥接合约 **锁定**资产（Lock）  
2. 验证者观察到锁定事件并提交证明/签名  
3. 链 B 铸造映射资产给用户（Mint）

### 反向赎回（B → A）
1. 用户在链 B **销毁**映射资产（Burn）  
2. 提交证明/签名到链 A  
3. 链 A **解锁**并释放原资产（Unlock）

## 5.2 常见安全机制
- **多签（t-of-n）**：资产释放需满足签名数量 ≥ t  
  $$
Valid \Leftrightarrow |signatures| \ge t
$$
- **门限签名（Threshold Signature）**：多方产生“一个聚合签名”  
  $$
\sigma = Combine(\sigma_1,\dots,\sigma_t)
$$
- **轻客户端验证桥**：接收链直接验证对方链区块头与 Merkle 证明，降低信任假设

> 现实中最常见事故：桥合约漏洞、验证者串谋、重放攻击、消息验证不严谨。


---
# Step 6：认识跨链安全威胁与防御机制

跨链安全可以用一个“安全三元组”来框定：

- **C**：跨链合约/证明系统（Cross-chain contract / proof system）  
- **V**：验证函数（Merkle / 签名 / 轻客户端）  
- **A**：原子性约束（时间锁 / 哈希锁 / 双向状态锁）  

安全性直觉：若验证通过，则动作必须满足原子性约束  
$$
\forall m \in M,\; V(C(m))=1 \Rightarrow A(m)=Atomic
$$

## 6.1 常见攻击与防御
1. **中继伪造**：伪造对方链状态/区块头  
   - 防御：轻客户端验证、门限签名、最终性检查  
2. **重放攻击**：旧消息/旧证明被重复使用  
   - 防御：Nonce、区块高度、通道序列号（IBC 里很常见）  
3. **桥合约漏洞**：权限控制/数学验证逻辑错误  
   - 防御：审计、最小权限、形式化验证、升级治理  
4. **资产双花/不一致**：锁定与铸造不同步  
   - 防御：双向证明、超时机制、原子性约束  
5. **验证者串谋**：t-of-n 验证者一起作恶  
   - 防御：去信任门限签名、随机验证者集、经济惩罚、分层验证

> 记住一句话：跨链的核心是“**可验证性**”与“**原子性**”，两者缺一不可。


---
# Step 7：实践 — HTLC 原子交换协议（Python 模拟）

下面我们用最小模型模拟两条链与 HTLC 合约。它不是生产级区块链，只用于理解流程与性质。

In [ ]:
from enum import Enum, auto

class ChainType(Enum):
    BITCOIN = auto()
    ETHEREUM = auto()
    POLYGON = auto()
    LIQUID = auto()

@dataclass
class Tx:
    kind: str
    data: Dict[str, Any]

@dataclass
class Block:
    height: int
    txs: List[Tx] = field(default_factory=list)
    timestamp: float = field(default_factory=time.time)

@dataclass
class Blockchain:
    chain_type: ChainType
    name: str
    accounts: Dict[str, float] = field(default_factory=dict)
    htlcs: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # contract_id -> contract
    mempool: List[Tx] = field(default_factory=list)
    blocks: List[Block] = field(default_factory=list)

    def height(self) -> int:
        return len(self.blocks)

    def create_account(self, addr: str, balance: float = 0.0):
        self.accounts.setdefault(addr, 0.0)
        self.accounts[addr] += balance

    def get_balance(self, addr: str) -> float:
        return self.accounts.get(addr, 0.0)

    def submit_tx(self, tx: Tx):
        self.mempool.append(tx)

    def mine_block(self):
        # 简化：挖矿把 mempool 全部打包并执行
        block = Block(height=self.height(), txs=self.mempool[:])
        for tx in block.txs:
            self._apply_tx(tx, block.height)
        self.mempool.clear()
        self.blocks.append(block)
        return block

    def _apply_tx(self, tx: Tx, current_height: int):
        k = tx.kind
        d = tx.data

        if k == "transfer":
            src, dst, amt = d["src"], d["dst"], d["amount"]
            if self.get_balance(src) < amt:
                raise ValueError(f"Insufficient funds: {src}")
            self.accounts[src] -= amt
            self.accounts[dst] = self.get_balance(dst) + amt

        elif k == "deploy_htlc":
            # 锁定资金到合约
            contract_id = d["contract_id"]
            sender, receiver, amt = d["sender"], d["receiver"], d["amount"]
            hashlock, timelock = d["hashlock"], d["timelock"]  # timelock: 区块高度截止
            if self.get_balance(sender) < amt:
                raise ValueError("Insufficient funds for HTLC")
            self.accounts[sender] -= amt
            self.htlcs[contract_id] = {
                "sender": sender,
                "receiver": receiver,
                "amount": amt,
                "hashlock": hashlock,
                "timelock": timelock,
                "redeemed": False,
                "refunded": False,
                "secret": None,
            }

        elif k == "redeem_htlc":
            contract_id, secret = d["contract_id"], d["secret"]
            c = self.htlcs[contract_id]
            if c["redeemed"] or c["refunded"]:
                raise ValueError("Contract already settled")
            if current_height > c["timelock"]:
                raise ValueError("Timelock expired; redeem not allowed")
            if sha256(secret) != c["hashlock"]:
                raise ValueError("Invalid secret")
            # 支付给 receiver
            self.accounts[c["receiver"]] = self.get_balance(c["receiver"]) + c["amount"]
            c["redeemed"] = True
            c["secret"] = secret

        elif k == "refund_htlc":
            contract_id = d["contract_id"]
            c = self.htlcs[contract_id]
            if c["redeemed"] or c["refunded"]:
                raise ValueError("Contract already settled")
            if current_height <= c["timelock"]:
                raise ValueError("Timelock not yet expired")
            # 退款给 sender
            self.accounts[c["sender"]] = self.get_balance(c["sender"]) + c["amount"]
            c["refunded"] = True

        else:
            raise ValueError(f"Unknown tx kind: {k}")


## 7.1 AtomicSwap 协议实现（发起 / 完成 / 超时退款）

In [ ]:
@dataclass
class AtomicSwap:
    chain_a: Blockchain
    chain_b: Blockchain
    swaps: Dict[str, Dict[str, Any]] = field(default_factory=dict)

    def initiate_swap(self, initiator: str, responder: str, amount_a: float, amount_b: float,
                      timelock_a: int = 10, timelock_b: int = 6) -> Tuple[str, str]:
        # 确保 timelock_b < timelock_a
        if timelock_b >= timelock_a:
            raise ValueError("timelock_b must be < timelock_a")

        secret = secrets.token_bytes(32)
        hashlock = sha256(secret)

        swap_id = sha256(secrets.token_bytes(16))
        # 以当前高度为基准，设置截止高度
        T_A = self.chain_a.height() + timelock_a
        T_B = self.chain_b.height() + timelock_b

        # 部署双方 HTLC
        cA = f"{swap_id}_A"
        cB = f"{swap_id}_B"

        self.chain_a.submit_tx(Tx("deploy_htlc", {
            "contract_id": cA,
            "sender": initiator,
            "receiver": responder,  # Bob 领取 Alice 的资产
            "amount": amount_a,
            "hashlock": hashlock,
            "timelock": T_A,
        }))

        self.chain_b.submit_tx(Tx("deploy_htlc", {
            "contract_id": cB,
            "sender": responder,
            "receiver": initiator,  # Alice 领取 Bob 的资产
            "amount": amount_b,
            "hashlock": hashlock,
            "timelock": T_B,
        }))

        self.swaps[swap_id] = {
            "initiator": initiator,
            "responder": responder,
            "amount_a": amount_a,
            "amount_b": amount_b,
            "hashlock": hashlock,
            "secret": secret,
            "contract_a": cA,
            "contract_b": cB,
            "T_A": T_A,
            "T_B": T_B,
        }
        return swap_id, hashlock

    def complete_swap(self, swap_id: str, secret: bytes):
        s = self.swaps[swap_id]
        # Alice 先在链 B 赎回，公开 secret
        self.chain_b.submit_tx(Tx("redeem_htlc", {"contract_id": s["contract_b"], "secret": secret}))
        self.chain_b.mine_block()

        # Bob 观察到 secret 后在链 A 赎回
        self.chain_a.submit_tx(Tx("redeem_htlc", {"contract_id": s["contract_a"], "secret": secret}))
        self.chain_a.mine_block()

    def refund_if_needed(self, swap_id: str):
        s = self.swaps[swap_id]
        # 任何一方都可以在超时后发起退款（此处简单演示）
        # 先尝试退款链B，再链A
        try:
            self.chain_b.submit_tx(Tx("refund_htlc", {"contract_id": s["contract_b"]}))
            self.chain_b.mine_block()
        except Exception:
            pass
        try:
            self.chain_a.submit_tx(Tx("refund_htlc", {"contract_id": s["contract_a"]}))
            self.chain_a.mine_block()
        except Exception:
            pass


## 7.2 运行 Demo：成功的原子交换

In [ ]:
def demo_atomic_swap_success():
    bitcoin = Blockchain(ChainType.BITCOIN, "Bitcoin主网")
    ethereum = Blockchain(ChainType.ETHEREUM, "Ethereum主网")

    alice_btc = "alice_btc"
    bob_eth = "bob_eth"

    # 初始化余额（为了展示：Alice 有 BTC，Bob 有 ETH）
    bitcoin.create_account(alice_btc, 2.0)
    ethereum.create_account(bob_eth, 20.0)

    print("=== 初始余额 ===")
    print("Alice BTC:", bitcoin.get_balance(alice_btc))
    print("Bob   ETH:", ethereum.get_balance(bob_eth))

    swap = AtomicSwap(bitcoin, ethereum)
    swap_id, h = swap.initiate_swap(
        initiator=alice_btc,
        responder=bob_eth,
        amount_a=1.0,
        amount_b=15.0,
        timelock_a=10,
        timelock_b=6
    )

    # 双链挖矿确认部署
    bitcoin.mine_block()
    ethereum.mine_block()

    secret = swap.swaps[swap_id]["secret"]
    swap.complete_swap(swap_id, secret)

    print("\n=== 交换完成后的余额 ===")
    print("Alice BTC:", bitcoin.get_balance(alice_btc), "（减少 1.0）")
    print("Bob   ETH:", ethereum.get_balance(bob_eth), "（减少 15.0）")

    # 在这个最小模型中，Alice 的 ETH 和 Bob 的 BTC 是通过 HTLC 直接转账实现的
    print("Alice ETH:", ethereum.get_balance(alice_btc), "（增加 15.0）")
    print("Bob   BTC:", bitcoin.get_balance(bob_eth), "（增加 1.0）")

demo_atomic_swap_success()


## 7.3 运行 Demo：超时退款（观察 time_lock 的意义）

In [ ]:
def demo_atomic_swap_timeout_refund():
    chain_a = Blockchain(ChainType.BITCOIN, "ChainA")
    chain_b = Blockchain(ChainType.ETHEREUM, "ChainB")

    alice = "alice"
    bob = "bob"

    chain_a.create_account(alice, 2.0)
    chain_b.create_account(bob, 20.0)

    swap = AtomicSwap(chain_a, chain_b)
    swap_id, _ = swap.initiate_swap(alice, bob, amount_a=1.0, amount_b=15.0, timelock_a=4, timelock_b=2)

    chain_a.mine_block()
    chain_b.mine_block()

    # 假设 Alice 不去链 B redeem，Bob 也无法拿到 secret
    # 我们推进区块高度直到超时
    chain_b.mine_block()  # height 2
    chain_b.mine_block()  # height 3 -> 超过 timelock_b(=2)
    chain_a.mine_block()  # height 2
    chain_a.mine_block()  # height 3
    chain_a.mine_block()  # height 4 -> 超过 timelock_a(=4) 还需 > 才能退款

    # 再挖一个块确保 > timelock_a
    chain_a.mine_block()  # height 5

    swap.refund_if_needed(swap_id)

    print("Alice 在链A余额:", chain_a.get_balance(alice), "（应退款回到 2.0）")
    print("Bob   在链B余额:", chain_b.get_balance(bob), "（应退款回到 20.0）")

demo_atomic_swap_timeout_refund()


### 7.4 练习
1. 把 `timelock_b` 改大（接近或超过 `timelock_a`），看看为什么会报错或产生风险。
2. 模拟 Bob 恶意拖延：Alice 在链 B redeem 后，Bob 是否总能在链 A 及时 redeem？

---
# Step 8：实践 — 跨链桥接（Lock-Mint）与侧链 Peg-in/Peg-out（Python 模拟）

## 8.1 跨链桥：Lock → Mint / Burn → Unlock（带阈值签名数量 t）

In [ ]:
@dataclass
class CrossChainBridge:
    chain_src: Blockchain
    chain_dst: Blockchain
    # 简化：用一个“验证者签名计数”来模拟 t-of-n
    threshold_t: int = 2
    n_validators: int = 3
    locks: Dict[str, Dict[str, Any]] = field(default_factory=dict)

    def lock_assets(self, user_src: str, amount: float, user_dst: str) -> str:
        bridge_id = sha256(secrets.token_bytes(16))
        # 锁定：在源链把钱从用户转到“桥地址”
        bridge_addr = f"bridge_{self.chain_src.name}"
        self.chain_src.create_account(bridge_addr, 0.0)

        self.chain_src.submit_tx(Tx("transfer", {"src": user_src, "dst": bridge_addr, "amount": amount}))
        self.locks[bridge_id] = {
            "user_src": user_src,
            "user_dst": user_dst,
            "amount": amount,
            "bridge_addr": bridge_addr,
            "minted": False,
            "burned": False,
            "signatures": 0,
        }
        return bridge_id

    def collect_signatures(self, bridge_id: str, k: int):
        # 模拟收集验证者签名
        self.locks[bridge_id]["signatures"] = min(self.n_validators, self.locks[bridge_id]["signatures"] + k)

    def mint_assets(self, bridge_id: str):
        info = self.locks[bridge_id]
        if info["minted"]:
            raise ValueError("Already minted")
        if info["signatures"] < self.threshold_t:
            raise ValueError(f"Not enough signatures: have {info['signatures']}, need {self.threshold_t}")
        # 目标链铸造：直接给用户加余额（模拟映射代币）
        self.chain_dst.create_account(info["user_dst"], 0.0)
        self.chain_dst.accounts[info["user_dst"]] += info["amount"]
        info["minted"] = True

    def burn_assets(self, bridge_id: str):
        info = self.locks[bridge_id]
        if not info["minted"]:
            raise ValueError("Not minted yet")
        if info["burned"]:
            raise ValueError("Already burned")
        # 用户在目标链销毁（减少余额）
        if self.chain_dst.get_balance(info["user_dst"]) < info["amount"]:
            raise ValueError("Insufficient mapped token to burn")
        self.chain_dst.accounts[info["user_dst"]] -= info["amount"]
        info["burned"] = True
        # 重新收集签名用于 unlock
        info["signatures"] = 0

    def unlock_assets(self, bridge_id: str):
        info = self.locks[bridge_id]
        if not info["burned"]:
            raise ValueError("Need burn first")
        if info["signatures"] < self.threshold_t:
            raise ValueError(f"Not enough signatures to unlock")
        # 在源链释放：从桥地址转回用户
        self.chain_src.submit_tx(Tx("transfer", {"src": info["bridge_addr"], "dst": info["user_src"], "amount": info["amount"]}))


## 8.2 Demo：桥接正向 + 反向（观察阈值签名 t 的影响）

In [ ]:
def demo_cross_chain_bridge():
    eth = Blockchain(ChainType.ETHEREUM, "Ethereum主网")
    polygon = Blockchain(ChainType.POLYGON, "Polygon网络")

    user_eth = "user_eth"
    user_polygon = "user_polygon"

    eth.create_account(user_eth, 10.0)

    bridge = CrossChainBridge(eth, polygon, threshold_t=2, n_validators=3)

    bridge_id = bridge.lock_assets(user_eth, 5.0, user_polygon)
    eth.mine_block()  # 确认锁定

    print("锁定后：用户ETH余额:", eth.get_balance(user_eth))

    # 收集签名并铸造
    bridge.collect_signatures(bridge_id, 2)
    bridge.mint_assets(bridge_id)

    print("铸造后：用户Polygon余额:", polygon.get_balance(user_polygon))

    # 反向赎回：burn -> 收集签名 -> unlock
    bridge.burn_assets(bridge_id)
    bridge.collect_signatures(bridge_id, 2)
    bridge.unlock_assets(bridge_id)
    eth.mine_block()

    print("赎回后：用户ETH余额:", eth.get_balance(user_eth))
    print("赎回后：用户Polygon余额:", polygon.get_balance(user_polygon))

demo_cross_chain_bridge()


### 8.3 练习
1. 把 `threshold_t` 改成 3，看看需要多少签名才能 mint/unlock。
2. 模拟验证者串谋：若 `t` 太小会发生什么？

## 8.4 侧链协议：Peg-in / Peg-out（主链 ↔ 侧链更强耦合）

In [ ]:
@dataclass
class SidechainPeg:
    main: Blockchain
    side: Blockchain
    peg_addr_main: str = field(init=False)
    peg_addr_side: str = field(init=False)

    def __post_init__(self):
        self.peg_addr_main = f"peg_{self.main.name}"
        self.peg_addr_side = f"peg_{self.side.name}"
        self.main.create_account(self.peg_addr_main, 0.0)
        self.side.create_account(self.peg_addr_side, 0.0)

    def peg_in(self, user_main: str, user_side: str, amount: float):
        # 主链锁定
        self.main.submit_tx(Tx("transfer", {"src": user_main, "dst": self.peg_addr_main, "amount": amount}))
        # 侧链释放/铸造等价资产给用户
        self.side.create_account(user_side, 0.0)
        self.side.accounts[user_side] += amount

    def peg_out(self, user_side: str, user_main: str, amount: float):
        # 侧链销毁/锁定
        if self.side.get_balance(user_side) < amount:
            raise ValueError("Insufficient side balance")
        self.side.accounts[user_side] -= amount
        # 主链释放
        self.main.submit_tx(Tx("transfer", {"src": self.peg_addr_main, "dst": user_main, "amount": amount}))


## 8.5 Demo：Bitcoin 主链 ↔ Liquid 侧链

In [ ]:
def demo_sidechain():
    btc = Blockchain(ChainType.BITCOIN, "Bitcoin主链")
    liquid = Blockchain(ChainType.LIQUID, "Liquid侧链")

    user_main = "user_btc"
    user_side = "user_lbtc"

    btc.create_account(user_main, 5.0)

    peg = SidechainPeg(btc, liquid)

    # Peg-in
    peg.peg_in(user_main, user_side, 2.0)
    btc.mine_block()

    print("Peg-in 后主链余额:", btc.get_balance(user_main))
    print("Peg-in 后侧链余额:", liquid.get_balance(user_side))

    # 侧链快速交易（简化为 transfer）
    merchant = "merchant_side"
    liquid.create_account(merchant, 0.0)
    liquid.submit_tx(Tx("transfer", {"src": user_side, "dst": merchant, "amount": 0.5}))
    liquid.mine_block()

    print("侧链交易后用户侧链余额:", liquid.get_balance(user_side))
    print("侧链交易后商户侧链余额:", liquid.get_balance(merchant))

    # Peg-out
    peg.peg_out(user_side, user_main, 1.0)
    btc.mine_block()

    print("Peg-out 后主链余额:", btc.get_balance(user_main))
    print("Peg-out 后侧链余额:", liquid.get_balance(user_side))

demo_sidechain()


### 8.6 对比理解：桥 vs 侧链
- **侧链**：与主链耦合更强（通常有固定的 peg 机制/验证组）
- **跨链桥**：更灵活支持异构链，但安全边界更复杂，需要严密的验证与治理


---
# Step 9：总结 — 数学基础与应用价值

## 9.1 核心数学技术清单（建议背下“关键词”）
- **哈希承诺**：不可逆性、碰撞阻力（用于哈希锁、Merkle、承诺）  
- **时间锁 / 原子性条件**：确保交换或回滚（HTLC）  
- **Merkle 树证明**：存在性证明（状态/事件）  
- **数字签名 / 门限签名**：验证者集合的授权与抗串谋  
- **（扩展）零知识证明**：在不暴露细节的情况下证明跨链状态（ZKBridge 等）

## 9.2 跨链本质：Trustless Interoperability
用一句形式化的话说：
$$
\forall Chain_i, Chain_j, \exists Proof: State_i \Rightarrow Valid\;on\;Chain_j
$$
即：只要能提供正确证明，链 j 就能验证链 i 的某个状态/事件成立。

## 9.3 典型应用场景
- DeFi：跨链资产流动、跨链抵押、跨链清算  
- 多链 DApp：不同链分工协作（结算层/执行层/数据可用性层）  
- Web3 基础设施：跨链消息总线、跨链身份、跨链治理  

## 9.4 机制对比速记
- **Atomic Swap**：HTLC（哈希锁 + 时间锁）  
- **Cosmos IBC**：轻客户端验证 + Merkle 证明  
- **ZKBridge**：零知识证明（证明对方链状态/事件）

---

### 最后思考（建议写几行回答）
1. 如果你要设计一个“学习中心跨链实验”，你会选 HTLC / 轻客户端 / 桥 哪一种作为主线？为什么？  
2. 如何在“可用性（速度/体验）”与“去信任化（安全）”之间做权衡？  
3. 你认为跨链桥最难的部分是：合约安全、证明验证、还是治理与运营？
